### Ingest Sales Report

#### Step 1: Load shared environment settings
This cell runs a shared setup notebook that defines reusable variables and helper functions for the ingestion flow.

What this step is intended to do:
* Load catalog, schema, volume, and folder variables
* Reuse helper functions such as audit or metadata column logic
* Keep this notebook focused on ingestion instead of repeating setup code


#### Step 2: Build the source file path
This section creates the full path to the CSV file that will be ingested.

What this step is intended to do:
* Combine the configured storage location pieces into one full path
* Point to the dated file that should be loaded into the bronze layer
* Make the source path easy to inspect before reading data


#### Step 3: Define the expected schema
This section declares the structure of the incoming CSV so the file is read with the correct column names and data types.

What this step is intended to do:
* Control how Spark interprets each column
* Enforce timestamps, numeric fields, and text fields consistently
* Reduce schema drift from CSV auto-detection


#### Step 4: Load and enrich the CSV data
This section reads the CSV file into a Spark DataFrame and applies any standard metadata columns needed for downstream processing.

What this step is intended to do:
* Load the raw event items file into Spark
* Apply the predefined schema during ingestion
* Add standard metadata columns for lineage or auditing


#### Step 5: Validate the target location
This section checks the destination naming components before writing the bronze table.

What this step is intended to do:
* Confirm the bronze database or schema reference
* Preview the destination path or table namespace
* Reduce mistakes before persisting the final table


#### Step 6: Persist the bronze table
This final section writes the ingested DataFrame to a Delta table in the bronze layer.

What this step is intended to do:
* Save the transformed DataFrame as a managed Delta table
* Overwrite the target when rerunning the ingestion
* Make the bronze dataset available for downstream silver transformations


In [ ]:
catalog = dbutils.widgets.get("catalog")
container = dbutils.widgets.get("container")
storage_account = dbutils.widgets.get("storage_account")

In [ ]:
%run "../setup/002_Environment_variables"

In [ ]:
dbutils.widgets.text('id'," ")
batch_id = dbutils.widgets.get('id')
batch_id

In [ ]:
# Build the full source path for the CSV file to ingest from cloud storage.
# This keeps the input file location parameterized through shared environment values.
path = f'/{volume}/{catalog}/{schema}/{file}/{batch_id}/timed_auction_*.csv'

In [ ]:
# Import explicit Spark SQL data types so the CSV schema can be declared manually.
from pyspark.sql.types import *

# Define the expected structure of the source CSV.
# Using an explicit schema prevents unwanted type inference issues.
schema_timedauction = StructType([
        StructField('Auction Name', StringType(), True),
        StructField('Auction Date', StringType(), True),
        StructField('Item', StringType(), True),
        StructField('Lot #', IntegerType(), True),
        StructField('Currency', StringType(), True),
        StructField('Current Bid', DoubleType(), True),
        StructField('Max Bid', DoubleType(), True),
        StructField('First Name', StringType(), True),
        StructField('Last Name', StringType(),),
        StructField('User email', StringType(), True),
        StructField('Date time', StringType(), True),
])

In [ ]:
# Read the source CSV into a Spark DataFrame using the predefined schema.
# The file is expected to contain a header row.
df_timedauction = spark.read.format('csv') \
     .option('multiLine', 'true') \
    .option("quote", '"') \
    .option("escape", '"') \
    .option('header', 'true') \
    .schema(schema_timedauction) \
    .load(path)

# Add standard metadata or audit columns using the shared helper function.
df_timedauction = add_columns(df_timedauction)

In [ ]:
df_timedauction = df_timedauction.withColumnsRenamed({col: col.replace(' ', '_').lower() for col in df_timedauction.columns})

In [ ]:
# Write the ingested DataFrame to the bronze layer as a Delta table.
# Overwrite mode is used so the table can be rebuilt on repeated runs.
df_timedauction \
    .write \
    .format('delta') \
    .mode('overwrite') \
    .partitionBy('batch_id') \
    .option('replaceWhere',f"batch_id = '{batch_id}'")\
    .saveAsTable(f'{catalog}.bronze.timedauction')